In [ ]:
import sqlite3
import pandas as pd
from rdkit import Chem
import sys
import os
from typing import Tuple, Optional
from tqdm import tqdm


def canonical_and_inchikey(smiles: str) -> Tuple[Optional[str], Optional[str]]:
    if smiles is None or (isinstance(smiles, float) and pd.isna(smiles)):
        return None, None
    s = str(smiles).strip()
    if s == "" or s.upper() == "N/A":
        return None, None
    try:
        mol = Chem.MolFromSmiles(s)
        if mol is None:
            return None, None
        can = Chem.MolToSmiles(mol, canonical=True, isomericSmiles=True)
        try:
            ik = Chem.MolToInchiKey(mol)
        except Exception:
            ik = None
        return can, ik
    except Exception:
        return None, None

def init_seen_db(db_path: str):
    need_create = not os.path.exists(db_path)
    conn = sqlite3.connect(db_path, timeout=30)
    if need_create:
        cur = conn.cursor()
        cur.execute("""
            CREATE TABLE IF NOT EXISTS seen (
                canonical_smiles TEXT PRIMARY KEY,
                spectrumid TEXT
            );
        """)
        conn.commit()
    return conn

def batch_iterable(seq, size):
    for i in range(0, len(seq), size):
        yield seq[i:i+size]

def process(input_csv: str, output_csv: str, db_path: str,
            chunksize: int = 100000, drop_invalid: bool = True,
            show_progress: bool = True, index: bool = False):
    conn = init_seen_db(db_path)
    cur = conn.cursor()

    reader = pd.read_csv(input_csv, chunksize=chunksize, dtype=str)
    first_write = True
    total_processed = 0
    total_written = 0

    for chunk_idx, chunk in enumerate(reader, start=1):
        if "SMILES" not in chunk.columns:
            raise ValueError("no 'SMILES' in csv, plz verify column names (case-sensitive).")
        if "SPECTRUMID" not in chunk.columns:
            raise ValueError("no 'SPECTRUMID' in csv, plz verify column names (case-sensitive).")

        smiles_series = chunk["SMILES"].fillna("").astype(str)
        results = []
        if tqdm and show_progress:
            iterator = tqdm(smiles_series.iteritems(), total=len(smiles_series), desc=f"chunk {chunk_idx}")
            for _, s in iterator:
                results.append(canonical_and_inchikey(s))
        else:
            for s in smiles_series:
                results.append(canonical_and_inchikey(s))

        chunk["Canonical_SMILES"] = [r[0] for r in results]
        chunk["InChIKey"] = [r[1] for r in results]
        
        if drop_invalid:
            before = len(chunk)
            chunk = chunk[chunk["Canonical_SMILES"].notna() & (chunk["Canonical_SMILES"].str.strip() != "")]
            after = len(chunk)
        if chunk.empty:
            print(f"[INFO] chunk {chunk_idx}: 无有效 canonical SMILES，跳过。", file=sys.stderr)
            total_processed += len(results)
            continue

        rows_to_insert = list(zip(chunk["Canonical_SMILES"].tolist(), chunk["SPECTRUMID"].tolist()))

        batch_size = 800 
        newly_inserted_set = set() 

        for batch in batch_iterable(rows_to_insert, batch_size):
            cur.executemany("INSERT OR IGNORE INTO seen (canonical_smiles, spectrumid) VALUES (?, ?)", batch)
            conn.commit()

            batch_cans = [t[0] for t in batch]
            batch_sids = [t[1] for t in batch]
            for sub_cans in batch_iterable(batch_cans, 200):
                placeholders_c = ",".join("?" for _ in sub_cans)
                placeholders_s = ",".join("?" for _ in batch_sids)
                sql = f"SELECT canonical_smiles, spectrumid FROM seen WHERE canonical_smiles IN ({placeholders_c}) AND spectrumid IN ({placeholders_s})"
                params = sub_cans + batch_sids
                cur.execute(sql, params)
                rows = cur.fetchall()
                for can, sid in rows:
                    # only add those where this sid corresponds to this canonical in our batch (i.e. newly inserted)
                    newly_inserted_set.add(can)

        # Filter chunk only keep rows with canonical in newly_inserted_set
        if newly_inserted_set:
            mask = chunk["Canonical_SMILES"].isin(newly_inserted_set)
            out_chunk = chunk[mask].copy()
        else:
            out_chunk = chunk.iloc[0:0]  # empty

        # write out_chunk to CSV
        if not out_chunk.empty:
            if first_write:
                out_chunk.to_csv(output_csv, index=index, mode='w', encoding='utf-8')
                first_write = False
            else:
                out_chunk.to_csv(output_csv, index=index, header=False, mode='a', encoding='utf-8')

            total_written += len(out_chunk)
        total_processed += len(results)

        print(f"[INFO] chunk {chunk_idx}: processed {len(results)} rows, wrote {len(out_chunk)} unique new rows (total written: {total_written})", file=sys.stderr)

    conn.close()
    print(f"[DONE] total processed rows: {total_processed}, total unique written rows: {total_written}. Output -> {output_csv}", file=sys.stderr)


input_csv = "cleaned_metadata.csv"
output_csv = "with_canonical_inchikey.csv"

df = pd.read_csv(input_csv)

tqdm.pandas(desc="canonicalize SMILES & generate InChIKey")

def process_smiles(smiles):
    try:
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return pd.Series([None, None])
        canonical = Chem.MolToSmiles(mol, canonical=True)
        inchikey = Chem.MolToInchiKey(mol)
        return pd.Series([canonical, inchikey])
    except Exception:
        return pd.Series([None, None])

df[["Canonical_SMILES", "InChIKey"]] = df["SMILES"].progress_apply(process_smiles)
df = df.dropna(subset=["Canonical_SMILES", "InChIKey"])
df.to_csv(output_csv, index=False)

<>:164: SyntaxWarning: invalid escape sequence '\L'
<>:165: SyntaxWarning: invalid escape sequence '\L'
<>:164: SyntaxWarning: invalid escape sequence '\L'
<>:165: SyntaxWarning: invalid escape sequence '\L'
C:\Users\1\AppData\Local\Temp\ipykernel_34232\3975521826.py:164: SyntaxWarning: invalid escape sequence '\L'
  input_csv = "E:\LD\csv\cleaned_metadata.csv"
C:\Users\1\AppData\Local\Temp\ipykernel_34232\3975521826.py:165: SyntaxWarning: invalid escape sequence '\L'
  output_csv = "E:\LD\csv\with_canonical_inchikey.csv"
标准化SMILES & 生成InChIKey中:   0%|          | 1811/1632310 [00:01<18:26, 1474.04it/s][09:57:14] bond type above 3 (17) is treated as unspecified!
[09:57:14] bond type above 3 (17) is treated as unspecified!
[09:57:14] bond type above 3 (17) is treated as unspecified!
[09:57:14] Invalid InChI prefix in generating InChI Key
[09:57:14] bond type above 3 (17) is treated as unspecified!
[09:57:14] bond type above 3 (17) is treated as unspecified!
[09:57:14] bond type above 3 (

✅ 完成！输出文件：E:\LD\csv\with_canonical_inchikey.csv
共保留 1632310 条有效记录。
